<a href="https://colab.research.google.com/github/ncsu-geoforall-lab/GIS582-assignments/blob/main/5AB%20-%20Geomorphology/5B_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 5B: Geomorphometry I: Terrain Modeling

**Course:** [GIS 582 - Geospatial Modeling and Analysis](https://ncsu-geoforall-lab.github.io/geospatial-modeling-course/grass/terrain_modeling.html)  
**Institution:** [NC State University, Center for Geospatial Analytics](https://cnr.ncsu.edu/geospatial/)
**Instructors:** Helena Mitasova, Corey White, and team

## Learning Objectives

- summary parameters: volumes, surface areas
- first and second order point parameters: general approach
- methods for slope, aspect and curvatures using polynomial and spline approximation
- combining parameters to map landforms and terrain features
- computing parameters from noisy data, accuracy and uncertainty, scale and level of detail
- raster time series analysis, quantification of coastal change

## Tutorial Outline

* Part 1: Environment Setup
* Part 2: Compute basic topographic parameters: slope and aspect
* Part 3: Compute slope along road
* Part 4: Curvatures
* Part 5: Landforms
* Part 6: Optional - Raster time series analysis
* Part 7: Optional - Cut and fill and volume

---
## Part 1: Environment Setup

### Install GRASS

**Important:** This setup takes 3-5 minutes. You'll need to run it each time you start a new Colab session.

In [ ]:
!add-apt-repository -y ppa:ubuntugis/ubuntugis-unstable
!apt update
!apt-get install -y grass-core grass-dev pdal

Check that GRASS is installed and available:

In [ ]:
!grass --version

Check which Python version is running:

In [ ]:
import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import GRASS Python packages

We need to locate GRASS Python packages using `grass --config python_path` and add them to `sys.path`.

In [ ]:
import subprocess
import os
from pathlib import Path

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj

### Download North Carolina Sample Dataset

In [ ]:
!grass --tmp-project XY --exec g.download.project url=https://grass.osgeo.org/sampledata/north_carolina/nc_spm_08_grass7.tar.gz path=/content

### Initialize GRASS Session

In [ ]:
# Start GRASS session
grassdata = "/content"
project = "nc_spm_08_grass7"
mapset = "user1"  # Create a new mapset for our work

# Start GRASS Session
session = gj.init(Path(project, mapset))
print("GRASS session started.")

---
## Part 2: Compute basic topographic parameters: slope and aspect

Compute the slope and aspect of the terrain using `r.slope.aspect` tool.

In [ ]:
gs.run_command("g.region", raster="elevation", flags="p")
gs.run_command(
    "r.slope.aspect",
    elevation="elevation",
    slope="myslope",
    aspect="myaspect",
    overwrite=True
)

gs.run_command("r.colors", map="myaspect", color="aspectcolr")

To explore computation of basic topographic parameters we set our computational region to NW section of our elevation DEM and compute steepest slope and aspect maps.

In [ ]:
gs.run_command("g.region", raster="elevation", s="s+7000", e="e-7000", flags="p")
print("Region set to 7000 units inside the elevation raster bounds.")

# Slope map
slope_map = gj.Map(filename="myslope.png")
slope_map.d_rast(map="myslope")
slope_map.d_legend(raster="myslope", at=(2,40,2,6))
slope_map.show()

# Aspect map
aspect_map = gj.Map(filename="myaspect.png")
aspect_map.d_rast(map="myaspect")
aspect_map.d_legend(raster="myaspect", at=(2, 40, 2, 6))
aspect_map.show()

DEMs are sometimes distributed with elevations stored as **integer** values, changing the pattern of topographic parameters. To show impact of integer elevation values in meters on slope and aspect pattern we first compute an integer DEM, its slope and aspect and then compare the resulting maps with the original floating point DEM results

Compute integer DEM and derive its slope and aspect.

In [ ]:
gs.run_command("r.mapcalc", expression="elev_int = int(elevation)")
gs.run_command(
    "r.slope.aspect",
    elevation="elev_int",
    aspect="aspect_int_10m",
    slope="slope_int_10m",
)
gs.run_command("r.colors", map="aspect_int_10m", color="aspectcolr")

In [ ]:
# Slope map
slope_map = gj.Map(filename="slope_intmap.png")
slope_map.d_rast(map="slope_int_10m")
slope_map.d_legend(raster="slope_int_10m", at=(2,40,2,6))
slope_map.show()

# Aspect map
aspect_map = gj.Map(filename="aspect_intmap.png")
aspect_map.d_rast(map="aspect_int_10m")
aspect_map.d_legend(raster="aspect_int_10m", at=(2, 40, 2, 6))
aspect_map.show()

View the slope and aspect maps in an interactive map viewer.

In [ ]:
myslope_map_interactive = gj.InteractiveMap()

# Set opacity to 1.0 (default is 0.8) so that we can see colors clearly
myslope_map_interactive.add_raster("myslope", opacity=1.0)
myslope_map_interactive.add_raster("slope_int_10m", opacity=1.0)
myslope_map_interactive.add_layer_control()

myslope_map_interactive.show()


It is also helpful to compare the histograms.

Display historgram of slope (`myslope`, `slope_int_10m`) and aspect (`myaspect`, `aspect_int_10m`) maps.

In [ ]:
slope_int_10m = gj.Map(filename="histslope_int_10m.png")
slope_int_10m.d_histogram(map="slope_int_10m")
slope_int_10m.show()

In [ ]:
slope_map = gj.Map(filename="histmyslope.png")
slope_map.d_histogram(map="myslope")
slope_map.show()

In [ ]:
aspect_int_10m = gj.Map(filename="aspect_intmap.png")
aspect_int_10m.d_histogram(map="aspect_int_10m")
aspect_int_10m.show()

In [ ]:
aspect_map = gj.Map(filename="myaspect.png")
aspect_map.d_histogram(map="myaspect")
aspect_map.show()

#### Question 2.1

Can you explain the difference in slope and aspect maps and their histograms derived from integer (m vertical resolution) and floating point DEMs?

`Add your answer here to question 2.1 here.`

---
## Part 3: Compute slope along road

First set the region to the extent of the bus route #11 and to 10m resolution.


In [ ]:
gs.run_command("g.region", vect="busroute11", align="elevation", res=10, flags="p")

Then convert the vector line of the route to raster using the direction of the route.

In [ ]:
gs.run_command(
    "v.to.rast", input="busroute11", type="line", output="busroute11_dir", use="dir"
)
gs.run_command("r.colors", map="busroute11_dir", color="aspect")

Display the bus route direction raster map.

In [ ]:
busroute11_dir_map = gj.Map(filename="bus11direction.png")
busroute11_dir_map.d_rast(map="busroute11_dir")
busroute11_dir_map.d_legend(raster="busroute11_dir")
busroute11_dir_map.show()

Compute the steepest slope of the topography around the route by assigning the values of slope derived from a DEM.

In [ ]:
gs.run_command(
    "r.slope.aspect",
    elevation="elevation",
    slope="slope_dem",
    aspect="aspect_dem"
)

gs.run_command(
    "r.mapcalc", expression="route_slope_dem = if(busroute11_dir, slope_dem)"
)

Display the `route_slope_dem` map.

In [ ]:
route_slope_dem_map = gj.Map()
route_slope_dem_map.d_rast(map="route_slope_dem")
route_slope_dem_map.d_legend(raster="route_slope_dem")
route_slope_dem_map.show()

Then compute the slope in the direction of the route using the difference between aspect of the topography and the route direction angles. Display the results along with the elevation contours and compute univariate statistics.

In [ ]:
gs.run_command(
    "r.mapcalc",
    expression="route_slope_dir = abs(atan(tan(slope_dem) * cos(aspect_dem - busroute11_dir)))",
)
gs.run_command("r.colors", map="route_slope_dem,route_slope_dir", color="bcyr", flags="e")
gs.run_command("r.contour", input="elevation", output="contours", step=3)

Display the results along with the elevation contours and compute univariate statistics.

In [ ]:
route_slope_map = gj.Map(filename="route_slope_dem.png")
route_slope_map.d_rast(map="route_slope_dem")
route_slope_map.d_vect(map="contours")
route_slope_map.d_legend(raster="route_slope_dem")
route_slope_map.show()

route_slope_dir_map = gj.Map(filename="route_slope_dir.png")
route_slope_dir_map.d_rast(map="route_slope_dir")
route_slope_dir_map.d_vect(map="contours")
route_slope_dir_map.d_legend(raster="route_slope_dir")
route_slope_dir_map.show()

Examine the univariate statistics.

In [ ]:
!r.univar route_slope_dem
!r.univar route_slope_dir

#### Question 3.1

What is the difference between the two slope maps? Which slope is steeper on average and why?

---
## Part 4: Curvatures

Compute curvatures simultaneously with interpolation of DEM from lidar data using regularized spline with tension. Evaluate the influence of the spline parameters on the spatial pattern of the curvature maps.

In [ ]:
gs.run_command("g.region", region="rural_1m", flags="p")
gs.run_command(
    "v.surf.rst",
    input="elev_lid792_bepts",
    elevation="elev_lid_1m",
    aspect="asp_lid_1m",
    pcurvature="pc_lid_1m",
    tcurvature="tc_lid_1m",
    npmin=120,
    segmax=25,
)

Display the profile and tangential curvatures as 2D images.

In [ ]:
profcurvature1_map = gj.Map(filename="profcurvature_dtm.png")
profcurvature1_map.d_rast(map="pc_lid_1m")
profcurvature1_map.d_legend(raster="pc_lid_1m")
profcurvature1_map.show()

tangcurvature1_map = gj.Map(filename="tangcurvature_dtm.png")
tangcurvature1_map.d_rast(map="tc_lid_1m")
tangcurvature1_map.d_legend(raster="tc_lid_1m")
tangcurvature1_map.show()

The curvature maps reflect the lidar survey pattern rather than topographic features.

We can lower the tension and increase the smoothing to reduce the noise and map the curvatures of topography.

In [ ]:
gs.run_command("g.region", region="rural_1m", flags="p")
gs.run_command(
    "v.surf.rst",
    input="elev_lid792_bepts",
    elevation="elev_lidt15_1m",
    aspect="asp_lidt15_1m",
    pcurvature="pc_lidt15_1m",
    tcurvature="tc_lidt15_1m",
    npmin=120,
    segmax=25,
    tension=15,
    smooth=1.0,
)

In [ ]:
profcurvature1_map = gj.Map(filename="profcurvature_dtm2.png")
profcurvature1_map.d_rast(map="pc_lidt15_1m")
profcurvature1_map.d_legend(raster="pc_lidt15_1m")
profcurvature1_map.show()

tangcurvature1_map = gj.Map(filename="tangcurvature_dtm2.png")
tangcurvature1_map.d_rast(map="tc_lidt15_1m")
tangcurvature1_map.d_legend(raster="tc_lidt15_1m")
tangcurvature1_map.show()

Optionally, you can display the results in 3D view by draping the curvature maps over DEMs as color.

#### Question 4.1

How does changing the spline parameters influence the spatial pattern of curvatures?

`Add your answer here`

---
## Part 5: Landforms

Extract landforms at different levels of detail by adjusting the size of moving window.

Set rural subregion at 1m resolution, compute landforms using 9m and 45m neighborhood: read the manual to learn more.

#### Question 5.1

Explain types of landforms and the role of the neighborhood size.

In [ ]:
gs.run_command("g.region", region="rural_1m", flags="p")
gs.run_command("r.param.scale", input="elev_lid792_1m", output="feature9c_1m", size=9, method="feature")
gs.run_command("r.param.scale", input="elev_lid792_1m", output="feature45c_1m", size=45, method="feature")

Display with legend, save images for the report.

> Optionally display the feature maps draped over `elev_lid792_1m` as color.

In [ ]:
feature9c_1m_map = gj.Map(filename="landforms9.png")
feature9c_1m_map.d_rast(map="feature9c_1m")
feature9c_1m_map.d_legend(raster="feature9c_1m", at=(2, 20, 2, 6))
feature9c_1m_map.show()

In [ ]:
landforms45_map = gj.Map(filename="landforms45.png")
landforms45_map.d_rast(map="feature45c_1m")
landforms45_map.d_vect(map="elev_lid792_cont1m", color="grey")
landforms45_map.d_legend(raster="feature45c_1m", at=(2, 20, 2, 6))
landforms45_map.show()

Map basic landforms using geomorphons DSM, read the manual page and the associated paper to learn how it works.

In [ ]:
gs.run_command("r.geomorphon", elevation="elev_lid792_1m", forms="geomorphon_s50d10", search=50, dist=10)

Display the resulting raster map of geomorphons.

In [ ]:
geomorphon_map = gj.Map(filename="geomorphon_s50d10.png")
geomorphon_map.d_rast(map="geomorphon_s50d10")
geomorphon_map.d_vect(map="elev_lid792_cont1m", color="grey")
geomorphon_map.d_legend(raster="geomorphon_s50d10", at=(2, 20, 2, 8), flags="c")
geomorphon_map.show()

#### Question 5.2

Apply geomorphon to the 10m resolution DEM "elevation" (set region to raster elevation) and explain what is shown in the resulting map. Include the output map in your report.

In [ ]:
# Add code for question 5.2 here

`Add response to question 5.2 here`

---
## Part 6: Optional - Raster time series analysis

We will explore elevation time series analysis using a series of coastal DEMs provided at [NagsHead_series Mapset](http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data/NagsHead_series.zip).

In [ ]:
!curl -L http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data/NagsHead_series.zip -o /tmp/NagsHead_series.zip && unzip -q /tmp/NagsHead_series.zip -d /content/nc_basic_spm_grass7

Change the mapset to NagsHead_series and set the region to the extent of the DEMs.

> If you don't see the mapset, make sure you downloaded it and unzipped it correctly.

In [ ]:
# Session was define when we initated GRASS
session.switch_mapset("NagsHead_series")
gs.run_command("g.region", raster="NH_2008_1m", flags="p")

View the `NH_2008_1m` DEM.

In [ ]:
NH_2008_1m_map = gj.Map()
NH_2008_1m_map.d_rast(map="NH_2008_1m")
NH_2008_1m_map.d_legend(raster="NH_2008_1m", at=(2, 50, 2, 6))
NH_2008_1m_map.show()

Run the series analysis and explain the results:

In [ ]:
gs.run_command("g.region", raster="NH_2008_1m", flags="p")
series_methods = [('min','minimum'),('max','maximum'),('mintime','min_raster'),('maxtime','max_raster'),('range','range'),('avg','average'),('stddev','stddev')]
for m in series_methods:
    posttext = m[0]
    method = m[1]
    output=f"NH_9908_{posttext}"
    gs.run_command(
        'r.series',
        input="NH_1999_1m,NH_2001_1m,NH_2004_1m,NH_2005_1m,NH_2007_1m,NH_2008_1m",
        out=output,
        method=method
    )
    print(output)

In [ ]:
# Set the color table
!printf '%s\n' "0 grey" "0.5 cyan" "1.0 yellow" "2 orange" "5 red" "15 magenta" | r.colors NH_9908_stddev rules=-

NH_9908_stddev_map = gj.Map()
NH_9908_stddev_map.d_rast(map="NH_9908_stddev")
NH_9908_stddev_map.d_legend(raster="NH_9908_stddev", at=(2,50,2,6))
NH_9908_stddev_map.show()

In [ ]:
!r.colors -n NH_9908_range color=inferno

NH_9908_range_map = gj.Map(filename="NH_9908_range.png")
NH_9908_range_map.d_rast(map="NH_9908_range")
NH_9908_range_map.d_legend(raster="NH_9908_range", at=(2,50,2,6))
NH_9908_range_map.show()

#### Question 6.1

Which maps are core and envelope? Which landforms have high standard deviation and what does it mean?

`Add response to question 6.1 here`

> You can use cutting plane in 3D view to show the core and envelope.
Add constant elevation plane at -1m for reference, set zexag to 3-5 (the default is too high).
Assign surfaces constant color, use top or bottom surface for crossection color.
When using top for color, lower the light source to make top surface dark and highlight the crossection. Include the output maps and the 3D view with a cutting plane in your report to illustrate the answers to the two questions.

## Part 7: Optional - Cut and fill and volume

Compute cut and fill for 4m deep excavation to build a facility. First, set the region and display facility on top of orthophoto.

In [ ]:
gs.run_command("g.region", region="rural_1m", flags="p")
ortho_2001_t792_1m_map = gj.Map()
ortho_2001_t792_1m_map.d_rast(map="ortho_2001_t792_1m")
ortho_2001_t792_1m_map.d_rast(map="facility")
ortho_2001_t792_1m_map.show()

Then set (raster) mask to the facility map and find minimum elevation within the facility:

In [ ]:
gs.run_command("r.mask", raster="facility")

In [ ]:
gs.run_command("r.univar", map="elevation")

Minimum which you obtain should be 123.521m. Bottom of 4m excavation will be then

123.52 - 4 = 119.52


Use raster algebra to create the excavation:

In [ ]:
gs.run_command("r.mapcalc", expression="excavation = elevation - 119.52")
gs.run_command("r.univar", map="excavation")

Display the `excavation` map.

In [ ]:
excavation_map = gj.Map()
excavation_map.d_rast(map="excavation")
excavation_map.show()

Minimum you get should be 4.00057 and maximum 9.50554. Note that the excavation is limited by the mask we set earlier, so we can now do global operation to compute the volume which applies just the the facility.

In [ ]:
gs.parse_command("r.volume", input="excavation")

Now remove mask. This is important so that your future work is not affected.

In [ ]:
gs.run_command("r.mask", flags="r")